# PyYAML — introductory notebook

M3 requires a minimal notebook per selected tool. This one covers **PyYAML**, used because
the project treats configuration as data: the dataset registry, the evaluation protocol
and the corruption ladder are YAML files that can be reviewed in a diff instead of being
buried in code.

The notebook loads the registry, validates the ladder, and shows that a bad file fails
loudly instead of silently producing an empty configuration.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

## Load the dataset registry

`load_registry` reads `data/datasets.yaml`, parses every dataset and pair, and rejects
duplicate ids. Keeping the checks in the loader means a typo in the YAML is an error at
load time, not a missing row three notebooks later.

In [2]:
from swsn.registry import load_registry

registry = load_registry("../data/datasets.yaml")
print("datasets:", len(registry.datasets), "| pairs:", len(registry.pairs))
print([d.id for d in registry.datasets.values()])

datasets: 9 | pairs: 18
['olist', 'instacart', 'movielens', 'craigslist_cars', 'cardekho_vehicles', 'airbnb_nyc', 'melbourne_housing', 'heart_failure', 'diabetes']


In [3]:
dataset = registry.datasets["heart_failure"]
print("class   :", dataset.cls.value)
print("slug    :", dataset.kaggle_slug)
print("licence :", dataset.licence)
print("files   :", list(dataset.sha256))

class   : c_nat
slug    : fedesoriano/heart-failure-prediction
licence : Database: Open Database, Contents: © Original Authors
files   : ['heart.csv']


## Load and validate the ladder

The ladder is a mapping from level id to a dimension and a list of operators. `load_ladder`
is a thin `yaml.safe_load`; the validation happens when a level is parsed, so an unknown
level or a malformed operator raises instead of returning `None`.

In [4]:
from swsn.corrupt import level_order, load_ladder, parse_level

ladder = load_ladder("../corruption/ladder.yaml")
for level in level_order(ladder):
    operators = [spec.id for spec in parse_level(ladder, level)]
    print(f"{level}: {len(operators):2d} operators  {operators}")

L0:  0 operators  []
L1:  8 operators  ['rename_synonym', 'rename_abbreviate', 'rename_translate', 'append_unit_to_header', 'autogenerated_name', 'duplicate_header', 'reorder_columns', 'add_spurious_column']
L2:  7 operators  ['date_reformat', 'number_thousand_separator', 'number_decimal_comma', 'currency_symbol', 'floatify', 'zero_pad', 'id_prefix']
L3:  6 operators  ['sentinel_null', 'mixed_type_in_column', 'whitespace_pad', 'unicode_variants', 'unit_in_value', 'case_variation_values']
L4:  3 operators  ['synonym_substitute', 'translate_values', 'category_recode']


L0 is deliberately empty: the uncorrupted baseline has no operator, and that is visible in
the data rather than implied by a comment.

## Fail loudly on a bad entry

The important behaviour is the failure, not the success. A schema file without `tables`
is rejected by `load_schema` with the path in the message, so a broken config names the
file that broke.

In [5]:
import pathlib
import tempfile

from swsn.schema_doc import load_schema

with tempfile.TemporaryDirectory() as tmp:
    bad = pathlib.Path(tmp) / "bad.yaml"
    bad.write_text("dataset: toy\n", encoding="utf-8")
    try:
        load_schema(bad)
    except ValueError as exc:
        print(f"{type(exc).__name__}: {str(exc).removeprefix(str(bad) + ': ')}")
    else:
        raise AssertionError("Expected load_schema to reject a schema without tables")

ValueError: schema must define `tables`


## Safe loading

`yaml.safe_load` constructs only plain Python types (dict, list, str, int, float, bool,
None). `yaml.load` without a loader can instantiate arbitrary objects, which is why the
project never uses it on files that come from outside the repository.

In [6]:
import yaml

sample = """\
version: "1"
threshold: 0.15
levels: [L0, L1]
"""
values = yaml.safe_load(sample)
print(values)
print({key: type(value).__name__ for key, value in values.items()})

{'version': '1', 'threshold': 0.15, 'levels': ['L0', 'L1']}
{'version': 'str', 'threshold': 'float', 'levels': 'list'}


Note `version: "1"` stays a string: the registry quotes versions on purpose, because
`version: 1.0` would load as a float and compare unequal to the string "1".

## Reproducing

```bash
make setup
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/tool_pyyaml.ipynb
```